In [ ]:
import pandas as pd
import ast
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

In [ ]:
# Load the CSV file
df = pd.read_csv('jargon_dictionary.csv')

# Convert the string representation of list to actual list
df['Simplified_List'] = df['Simplified_List'].apply(ast.literal_eval)

# Extract ONLY the first simplified word from each list
df['First_Simplified'] = df['Simplified_List'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)

# Create a readable version for display
df['Simplified'] = df['Simplified_List'].apply(
    lambda x: '; '.join([f"{i+1}. {word}" for i, word in enumerate(x)]) if isinstance(x, list) else x
)

print("Successfully converted 'Simplified_List' column from string to list.")
print(f"\nDataset contains {len(df)} jargon terms.")
print(f"\nFirst few entries:")
df[['Word', 'First_Simplified', 'Simplified']].head(10)

In [ ]:
# --- Basic Statistics (Using Only First Simplified Word) ---

# Calculate character length for jargon and first simplified word
df['jargon_length'] = df['Word'].str.len()
df['simplified_length'] = df['First_Simplified'].str.len()

# Calculate word count
df['jargon_word_count'] = df['Word'].str.split().str.len()
df['simplified_word_count'] = df['First_Simplified'].str.split().str.len()

# Calculate simplification ratio (using only first alternative)
df['simplification_ratio'] = df['simplified_length'] / df['jargon_length']

print("=== Basic Statistics (Based on First Simplified Word Only) ===")
print(f"\nAverage jargon term length: {df['jargon_length'].mean():.2f} characters")
print(f"Average simplified term length: {df['simplified_length'].mean():.2f} characters")
print(f"\nAverage jargon word count: {df['jargon_word_count'].mean():.2f} words")
print(f"Average simplified word count: {df['simplified_word_count'].mean():.2f} words")
print(f"\nMean simplification ratio: {df['simplification_ratio'].mean():.2f}")
print(f"(Values < 1 indicate shorter simplified terms)")

# Distribution of number of alternatives provided (informational only)
df['num_alternatives'] = df['Simplified_List'].apply(len)
print(f"\n=== Number of Alternatives Provided ===")
print(f"Average number of alternatives: {df['num_alternatives'].mean():.2f}")
print(f"Max alternatives for a single term: {df['num_alternatives'].max()}")
print(f"Terms with only 1 alternative: {(df['num_alternatives'] == 1).sum()}")
print(f"Terms with 2+ alternatives: {(df['num_alternatives'] > 1).sum()}")

In [ ]:
# --- Length Comparison Visualization (Using First Simplified Word) ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length comparison
axes[0].hist([df['jargon_length'], df['simplified_length']], 
             label=['Jargon', 'First Simplified'], bins=20, alpha=0.7)
axes[0].set_xlabel('Character Length')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Character Length Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Simplification ratio distribution
axes[1].hist(df['simplification_ratio'], bins=30, color='green', alpha=0.7)
axes[1].axvline(x=1, color='red', linestyle='--', label='No change (ratio=1)')
axes[1].set_xlabel('Simplification Ratio (simplified/jargon length)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Simplification Ratio Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTerms that got SHORTER: {(df['simplification_ratio'] < 1).sum()} ({(df['simplification_ratio'] < 1).sum() / len(df) * 100:.1f}%)")
print(f"Terms that got LONGER: {(df['simplification_ratio'] > 1).sum()} ({(df['simplification_ratio'] > 1).sum() / len(df) * 100:.1f}%)")
print(f"Terms same length: {(df['simplification_ratio'] == 1).sum()} ({(df['simplification_ratio'] == 1).sum() / len(df) * 100:.1f}%)")

In [ ]:
# --- Longest Jargon Terms (Using First Simplified Word) ---

print("\n=== 10 Longest Jargon Terms and Their First Simplified Alternative ===")
longest_jargon = df.nlargest(10, 'jargon_length')[['Word', 'First_Simplified', 'jargon_length', 'simplified_length']]
for idx, row in longest_jargon.iterrows():
    print(f"\n{row['Word']} ({row['jargon_length']} chars)")
    print(f"  → {row['First_Simplified']} ({row['simplified_length']} chars)")

In [ ]:
# --- Biggest Simplifications (Using First Simplified Word) ---

df['char_reduction'] = df['jargon_length'] - df['simplified_length']

print("\n=== Top 10 Terms with Biggest Character Reduction ===")
biggest_reductions = df.nlargest(10, 'char_reduction')[['Word', 'First_Simplified', 'char_reduction']]
for idx, row in biggest_reductions.iterrows():
    print(f"\n{row['Word']} → {row['First_Simplified']}")
    print(f"  Reduction: {row['char_reduction']} characters")

In [ ]:
# --- Most Frequent First Simplified Alternatives Analysis ---

# Count frequency of each first simplified word (lowercase for consistency)
first_simplified_clean = df['First_Simplified'].str.lower().str.strip()
first_alternative_counts = first_simplified_clean.value_counts()

print("\n=== Top 20 Most Frequent FIRST Simplified Alternatives ===")
print("(These are the primary simplified terms suggested most often)\n")

# Display as a bar chart
plt.figure(figsize=(12, 8))
top_20 = first_alternative_counts.head(20)
sns.barplot(x=top_20.values, y=top_20.index, palette="viridis")
plt.xlabel('Frequency (times used as first alternative)')
plt.ylabel('Simplified Term')
plt.title('Top 20 Most Frequent First Simplified Alternatives')
plt.tight_layout()
plt.show()

# Print the top 20 with counts
for i, (term, count) in enumerate(top_20.items(), 1):
    print(f"{i:2d}. '{term}' - used {count} times")

In [ ]:
# --- Word Count Analysis (Using First Simplified Word) ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count distribution
axes[0].hist([df['jargon_word_count'], df['simplified_word_count']], 
             label=['Jargon', 'First Simplified'], bins=range(1, 8), alpha=0.7, align='left')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word Count Distribution')
axes[0].legend()
axes[0].set_xticks(range(1, 8))
axes[0].grid(True, alpha=0.3)

# Scatter plot: jargon word count vs simplified word count
axes[1].scatter(df['jargon_word_count'], df['simplified_word_count'], alpha=0.5)
axes[1].plot([1, 7], [1, 7], 'r--', label='Equal word count')
axes[1].set_xlabel('Jargon Word Count')
axes[1].set_ylabel('First Simplified Word Count')
axes[1].set_title('Word Count Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nJargon terms with 1 word: {(df['jargon_word_count'] == 1).sum()}")
print(f"First simplified terms with 1 word: {(df['simplified_word_count'] == 1).sum()}")
print(f"\nSimplification typically reduces multi-word jargon to: {df[df['jargon_word_count'] > 1]['simplified_word_count'].mean():.2f} words")

In [ ]:
# --- Summary Statistics Table (Using First Simplified Word) ---

summary_stats = pd.DataFrame({
    'Metric': [
        'Total Terms',
        'Avg Jargon Length (chars)',
        'Avg First Simplified Length (chars)',
        'Avg Character Reduction',
        'Avg Jargon Word Count',
        'Avg First Simplified Word Count',
        'Mean Simplification Ratio',
        'Terms That Got Shorter (%)',
        'Terms That Got Longer (%)',
        'Most Common First Alternative'
    ],
    'Value': [
        len(df),
        f"{df['jargon_length'].mean():.2f}",
        f"{df['simplified_length'].mean():.2f}",
        f"{df['char_reduction'].mean():.2f}",
        f"{df['jargon_word_count'].mean():.2f}",
        f"{df['simplified_word_count'].mean():.2f}",
        f"{df['simplification_ratio'].mean():.2f}",
        f"{(df['simplification_ratio'] < 1).sum() / len(df) * 100:.1f}%",
        f"{(df['simplification_ratio'] > 1).sum() / len(df) * 100:.1f}%",
        f"'{first_alternative_counts.index[0]}' ({first_alternative_counts.iloc[0]}x)"
    ]
})

print("\n" + "="*60)
print("SUMMARY STATISTICS (Based on First Simplified Word Only)")
print("="*60)
print(summary_stats.to_string(index=False))
print("="*60)